In [62]:
import warnings
warnings.filterwarnings('ignore')

In [63]:
import os
import zipfile
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
from skimage.transform import resize
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [64]:
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

In [65]:
zip_path = "Y:\wassim-new-neuron\Training_data_inspection_validation_float16.zip"
real_data_zip_path = "/content/drive/MyDrive/MD4 2025-2026/Computer vision/real_data.zip"
extraction_path = "data_task_3"

data_path="data_task_3\Training_data_inspection_validation_float16"
csv_path = "data_task_3\Training_data_inspection_validation_float16\pipe_detection_label.csv"

In [11]:
os.makedirs(extraction_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_object:
  zip_object.extractall(extraction_path)

In [66]:
SEED = 42
SPLIT = 0.2

In [67]:
def load_dataset(data_path, full_image=True):


  df_labels = pd.read_csv(csv_path, sep=",")

  label_dict = dict(zip(df_labels["field_file"], df_labels["label"]))

  file_names = []
  images = []
  labels = []

  files = os.listdir(data_path)
  random.shuffle(files)

  for file in tqdm(files):
    if file.endswith(".npz") and file in label_dict:
      path = os.path.join(data_path, file)

      data = np.load(path)
      array = data["data"]
      if not full_image:
        array = array[:,:, 2]# Bz
        array = resize(array, (224, 224, 1), anti_aliasing=True)
      else:
        array = resize(array, (224, 224, 4), anti_aliasing=True)

      array = np.nan_to_num(array)
      if np.nanmax(array) != 0:
        array /= np.nanmax(array)

      images.append(array)
      labels.append(label_dict[file])
      file_names.append(file)


  X = np.array(images)
  y = np.array(labels)
  return X, y, file_names

In [33]:
X, Y, file_names = load_dataset(data_path= data_path)

100%|██████████| 4716/4716 [09:30<00:00,  8.26it/s]


In [80]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=SPLIT, random_state=42, stratify=Y)

In [81]:
import keras

In [82]:


data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip(mode="horizontal_and_vertical"),
        keras.layers.RandomRotation(factor=0.08),
        keras.layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
        keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1),
        
    ],
    name="data_augmentation",
)

In [83]:
def init_pipe_existance_classifier_model(full_image=True, use_augmentation=True):
  input_shape = (224, 224, 4) if full_image else (224, 224,1)

  model_layers = [keras.Input(shape=input_shape)]
  if use_augmentation:
    model_layers.append(data_augmentation)

  model_layers.extend([
      keras.layers.Conv2D(32, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      keras.layers.Conv2D(64, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      keras.layers.Conv2D(128, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      keras.layers.Flatten(),
      keras.layers.Dense(100),
      keras.layers.Dense(1, activation="sigmoid")
  ])

  pipe_existance_classifier_model = keras.Sequential(model_layers)
  return pipe_existance_classifier_model

In [84]:
pipe_existance_classifier_model = init_pipe_existance_classifier_model()

In [85]:
pipe_existance_classifier_model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 data_augmentation (Sequenti  (None, 224, 224, 4)      0         
 al)                                                             
                                                                 
 conv2d_12 (Conv2D)          (None, 222, 222, 32)      1184      
                                                                 
 max_pooling2d_12 (MaxPoolin  (None, 111, 111, 32)     0         
 g2D)                                                            
                                                                 
 conv2d_13 (Conv2D)          (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_13 (MaxPoolin  (None, 54, 54, 64)       0         
 g2D)                                                            
                                                      

In [86]:
pipe_existance_classifier_model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=10**(-4)), metrics=["accuracy"])

In [87]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
)

In [88]:
pipe_existance_classifier_model.fit(X_train, y_train, validation_split=0.2, batch_size=16, epochs=10_000, callbacks=[early_stopping])

Epoch 1/10000
189/189 [==============================] - 32s 142ms/step - loss: 0.6441 - accuracy: 0.6371 - val_loss: 0.6240 - val_accuracy: 0.6570
Epoch 2/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5989 - accuracy: 0.6845 - val_loss: 0.6114 - val_accuracy: 0.6702
Epoch 3/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5829 - accuracy: 0.7000 - val_loss: 0.6205 - val_accuracy: 0.6715
Epoch 4/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5742 - accuracy: 0.7067 - val_loss: 0.6017 - val_accuracy: 0.6808
Epoch 5/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5657 - accuracy: 0.7163 - val_loss: 0.5933 - val_accuracy: 0.6993
Epoch 6/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5568 - accuracy: 0.7229 - val_loss: 0.5895 - val_accuracy: 0.7033
Epoch 7/10000
189/189 [==============================] - 26s 140ms/step - loss: 0.5486 - accuracy: 0.7219 - val_

#test model


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [89]:
y_pred_probs = pipe_existance_classifier_model.predict(X_test)

30/30 [==============================] - 0s 7ms/step


In [90]:
y_pred_probs = pipe_existance_classifier_model.predict(X_test)

y_pred = (y_pred_probs>0.5).astype(int).flatten()


print(f"Accuracy:{accuracy_score(y_test, y_pred)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


30/30 [==============================] - 0s 12ms/step
Accuracy:0.855779427359491
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.81      0.82       377
           1       0.88      0.89      0.88       566

    accuracy                           0.86       943
   macro avg       0.85      0.85      0.85       943
weighted avg       0.86      0.86      0.86       943


Confusion Matrix:
[[306  71]
 [ 65 501]]


In [93]:
pipe_existance_classifier_model.save("task_03_model.keras")